![Week 3 Day 5](../../assets/banners/w3d5.png)

# Week 3 Exercise - Solution: Let Asha Edit a Question During the Review

On day 5 you watched GradeWise pause at `admin_review` and wait for Asha's yes or no. But real reviews aren't
yes-or-no. Asha reads the extracted exam and spots a typo in Q4a, or decides that Q3's definition should be worth
1.5 marks rather than 2. Today she shouldn't have to *reject* the exam and re-run the extraction (and pay for
it again!) over a small fix: she should be able to **edit the question right there, during the interrupt**.

You'll build a small review workflow around the real CS201 exam, **fully offline** (no LLM needed: the exam is
already extracted):

![Diagram](../../assets/diagrams/2397da0f2f44.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([start]) --> V[validate] --> A[[admin_review ⏸]]
  A -- "edited: check again" --> V
  A -- "bad edit: ask again" --> A
  A -- approved --> D[(store)]
  A -- rejected --> X([end])
```

</details>


Asha resumes with one of:
- `{"approved": True}` or `{"approved": False}`
- `{"question_edits": {"Q4a": {"text": "..."}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}}`

After an edit, the graph **re-validates** and **asks her again**, so she always approves exactly what gets stored.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">This is the reference solution. The packaged <code>gradewise/exam/graph.py</code> takes a simpler approach (a top-level <code>edits</code> dict merged into the whole paper); this version edits <b>one question, or one criterion, at a time</b>, and loops back for re-validation.</span>
        </td>
    </tr>
</table>

## Setup

In [ ]:
import copy
import json
import operator
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import ValidationError

from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.exam.summary import exam_summary, print_exam_summary
from gradewise.schemas import ExamPaper

EXAM = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
PAPER = EXAM.model_dump()  # 💡 a plain dict, as it is in the graph's state


def check(label: str, ok: bool) -> None:
    print(f"{'✅' if ok else '❌'} {label}")


def question(paper: dict, qid: str) -> dict:
    return next(q for q in paper["questions"] if q["id"] == qid)


print(f"Loaded {EXAM.exam_id}: {len(EXAM.questions)} questions, problems: {EXAM.problems() or 'none'}")

## Part 1: Applying edits to one question

First, a plain function, no graph yet. It must **never mutate** the paper it's given (state is shared!), and the
result must still be a valid `ExamPaper`.

In [ ]:
def apply_question_edits(paper: dict, question_edits: dict) -> tuple[dict, list[str]]:
    """Apply Asha's edits to a COPY of `paper`. Returns (new_paper, errors).

    question_edits looks like {"Q4a": {"text": "..."}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}
    """
    new = copy.deepcopy(paper)  # 💡 never mutate the state in place
    errors: list[str] = []
    for qid, changes in question_edits.items():
        question = next((q for q in new["questions"] if q["id"] == qid), None)
        if question is None:
            errors.append(f"No question {qid!r}. Valid IDs: {', '.join(q['id'] for q in new['questions'])}")
            continue
        for field, value in changes.items():
            if field != "criteria":
                question[field] = value
                continue
            for cid, patch in value.items():  # 💡 criteria are patched one by one, by ID
                criterion = next((c for c in question["criteria"] if c["id"] == cid), None)
                if criterion is None:
                    errors.append(f"{qid} has no criterion {cid!r}")
                else:
                    criterion.update(patch)
    if errors:
        return paper, errors
    try:
        return ExamPaper.model_validate(new).model_dump(), []  # 🔁 edits must still pass the schema
    except ValidationError as exc:
        return paper, [f"{'.'.join(map(str, e['loc']))}: {e['msg']}" for e in exc.errors()]

### ✅ Check your work

In [ ]:
fixed_text = "What is a race condition? Give an example."
new, errors = apply_question_edits(PAPER, {"Q4a": {"text": fixed_text}})
check("edits a question's text", question(new, "Q4a")["text"] == fixed_text and not errors)
check("leaves the original paper untouched", question(PAPER, "Q4a")["text"] != fixed_text)

new, errors = apply_question_edits(PAPER, {"Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}})
check("patches one criterion by ID", question(new, "Q3")["criteria"][0]["marks"] == 1.5)
check(
    "...and the paper now has a problem to show Asha",
    "Q3: criteria add up to 3.5" in " ".join(ExamPaper.model_validate(new).problems()),
)

new, errors = apply_question_edits(PAPER, {"Q99": {"text": "?"}})
check("reports an unknown question", bool(errors) and "Q99" in errors[0])

new, errors = apply_question_edits(PAPER, {"Q3": {"criteria": {"Q3.c9": {"marks": 1}}}})
check("reports an unknown criterion", bool(errors) and "Q3.c9" in errors[0])

new, errors = apply_question_edits(PAPER, {"Q5": {"max_marks": -1}})
check(
    "rejects an edit that breaks the schema, keeping the original",
    bool(errors) and question(new, "Q5")["max_marks"] == 5,
)

## Part 2: The review graph

The state, plus the two nodes that don't change: `validate` (day 3's check, using `ExamPaper.problems()`) and
`store`, which writes the approved exam to `output/week3_exercise_exam.json`.

In [ ]:
class ReviewState(TypedDict, total=False):
    paper: dict
    problems: list[str]
    approved: bool | None
    edited: bool
    log: Annotated[list[str], operator.add]


def validate(state: ReviewState) -> dict:
    problems = ExamPaper.model_validate(state["paper"]).problems()
    return {"problems": problems, "log": [f"validate → {problems or 'clean ✅'}"]}


def store(state: ReviewState) -> dict:
    out = OUTPUT_DIR / "week3_exercise_exam.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(state["paper"], indent=2, ensure_ascii=False), encoding="utf-8")
    return {"log": [f"store → {out.name}"]}

Now the heart of it: `admin_review` must handle **edits** as well as approve/reject, and `after_review` must send
an edited exam back to `validate`.

In [ ]:
def admin_review(state: ReviewState) -> dict:
    # ⚠️ No side effects before interrupt(): this node re-runs from the top on resume.
    paper = ExamPaper.model_validate(state["paper"])
    decision = interrupt({"summary": exam_summary(paper), "problems": state["problems"]})

    edits = decision.get("question_edits")
    if edits:
        new_paper, errors = apply_question_edits(state["paper"], edits)
        if errors:  # 💡 show Asha what went wrong, and ask again
            return {"edited": False, "approved": None, "problems": errors, "log": [f"review → edits refused: {errors}"]}
        return {"paper": new_paper, "edited": True, "approved": None, "log": [f"review → Asha edited {sorted(edits)}"]}
    approved = bool(decision.get("approved"))
    return {"approved": approved, "edited": False, "log": [f"review → {'approved ✅' if approved else 'rejected ❌'}"]}


def after_review(state: ReviewState) -> str:
    if state.get("edited"):
        return "validate"  # 🔁 re-check the edited exam, then show it to Asha again
    if state.get("approved") is None:
        return "admin_review"  # 💡 a refused edit: ask her again, with the errors as problems
    return "store" if state["approved"] else END

In [ ]:
from gradewise.display import show_graph

def build_review_graph():
    builder = StateGraph(ReviewState)
    for node in (validate, admin_review, store):
        builder.add_node(node.__name__, node)
    builder.add_edge(START, "validate")
    builder.add_edge("validate", "admin_review")
    builder.add_conditional_edges("admin_review", after_review, ["validate", "admin_review", "store", END])
    builder.add_edge("store", END)
    return builder.compile(checkpointer=InMemorySaver())  # ⚠️ interrupt() needs a checkpointer


review = build_review_graph()
show_graph(review)

## Part 3: Asha's review session

Round 1: the graph pauses with the clean exam. Asha makes two edits: she fixes Q4a's wording and lowers Q3's
definition criterion to 1.5 marks, but **forgets** to give the extra half mark to another criterion.

In [ ]:
config = {"configurable": {"thread_id": "asha-review"}}
result = review.invoke({"paper": PAPER, "log": []}, config)
print("⏸ Round 1. Problems shown to Asha:", result["__interrupt__"][0].value["problems"] or "none")

result = review.invoke(
    Command(resume={"question_edits": {"Q4a": {"text": fixed_text}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}}),
    config,
)
round_2 = result.get("__interrupt__")
print(
    "⏸ Round 2. Problems shown to Asha:", round_2[0].value["problems"] if round_2 else "(the graph didn't pause again)"
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before running on: where should the half mark go? Q3.c2 and Q3.c3 are one mark each for naming a deadlock condition. What edit makes the criteria add up to 4 again? And what do you predict the graph will do after she sends it: store, or pause again?</span>
        </td>
    </tr>
</table>

In [ ]:
if round_2:
    result = review.invoke(Command(resume={"question_edits": {"Q3": {"criteria": {"Q3.c2": {"marks": 1.5}}}}}), config)
round_3 = result.get("__interrupt__")
print("⏸ Round 3. Problems shown to Asha:", (round_3[0].value["problems"] or "none ✅") if round_3 else "(no pause)")
if round_3:
    print_exam_summary(ExamPaper.model_validate(review.get_state(config).values["paper"]))

In [ ]:
if round_3:
    result = review.invoke(Command(resume={"approved": True}), config)
print("\n".join(result.get("log", [])))

stored = OUTPUT_DIR / "week3_exercise_exam.json"
final = ExamPaper.model_validate_json(stored.read_text(encoding="utf-8")) if stored.exists() and round_3 else None
check("the graph paused three times (edit → fix → approve)", bool(round_2 and round_3))
check(
    "the stored exam has Asha's Q4a wording", bool(final) and question(final.model_dump(), "Q4a")["text"] == fixed_text
)
check("the stored exam is clean", bool(final) and final.problems() == [])

### One more path: a bad edit
What if Asha types a negative mark? The edit must be refused, and she must see why.

In [ ]:
config = {"configurable": {"thread_id": "asha-typo"}}
review.invoke({"paper": PAPER, "log": []}, config)
result = review.invoke(Command(resume={"question_edits": {"Q5": {"max_marks": -5}}}), config)

pending = result.get("__interrupt__")
shown = pending[0].value["problems"] if pending else []
print("Shown to Asha:", shown)
check("a bad edit pauses again with the error", bool(pending) and any("max_marks" in p for p in shown))
check("the paper was not changed", question(review.get_state(config).values["paper"], "Q5")["max_marks"] == 5)

## Bonus: the same idea in `week3/project/exam_graph.py`

The milestone script's `admin_review` has a `✋ TODO` for exactly this. Here's the change, reusing
`apply_question_edits` (paste the function into the script too):

```python
def admin_review(state: ExamState) -> dict:
    decision = interrupt({"paper": state.get("paper"), "problems": state.get("problems", [])})
    if decision.get("question_edits") and state.get("paper"):
        paper, errors = apply_question_edits(state["paper"], decision["question_edits"])
        if errors:
            return {"approved": False, "problems": errors}
        return {"paper": paper, "problems": ExamPaper.model_validate(paper).problems(),
                "approved": bool(decision.get("approved"))}
    return {"approved": bool(decision.get("approved")) and bool(state.get("paper"))}
```

💡 Unlike this notebook, the script doesn't loop back for a second look, so the edit and the approval
arrive together: `{"approved": True, "question_edits": {...}}`. Which design is better for Asha? The loop
costs her one more click, but guarantees she approves exactly what is stored.

## Recap

You gave the human in the loop real power: not just yes or no, but **targeted edits**, applied to a copy,
**re-validated** against the schema and the marking rules, and shown back to her before anything is stored.
That's the difference between a rubber stamp and a genuine review.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">"Reject and re-run" costs a model call and the reviewer's patience; "edit in place" costs neither. Small design choices like this decide whether busy teachers, or claims handlers, or auditors, actually <i>use</i> the AI system you build, or quietly go back to the spreadsheet.</span>
        </td>
    </tr>
</table>

⏭ **Next:** week 4, `week4/day1.ipynb`: agentic RAG, and reading Sam's handwriting.